# Data Generation
Synthetic dataset for utilisation and forecast analysis

In [83]:
import pandas as pd
import numpy as np

## Base structure

In [84]:
dates = pd.date_range(start="2026-01-01", end="2026-04-30", freq="B")
regions = ["North", "South", "East", "West"]
products = ["Mini Excavator","Skid Steer Loader","Telehandler","Dump Truck","Concrete Mixer"]

df = pd.MultiIndex.from_product(
    [dates, regions, products],
    names=["date", "region", "product"]
).to_frame(index=False)

## Utilisation model

In [85]:
df["month_num"] = df["date"].dt.month

seasonality_map = {1: -0.05, 2: -0.02, 3: 0.03, 4: 0.05}
df["seasonality"] = df["month_num"].map(seasonality_map)

region_effect = {"North": -0.03, "South": 0.04, "East": 0.0, "West": 0.02}
df["region_adj"] = df["region"].map(region_effect)

product_effect = {
    "Mini Excavator": 0.05,
    "Skid Steer Loader": 0.03,
    "Telehandler": 0.01,
    "Dump Truck": -0.01,
    "Concrete Mixer": -0.02
}
df["product_adj"] = df["product"].map(product_effect)

noise = np.random.normal(0, 0.015, len(df))
base = 0.27

df["utilisation"] = (
    base
    + df["seasonality"]
    + df["region_adj"]
    + df["product_adj"]
    + noise
)

df["utilisation"] = df["utilisation"].clip(lower=0)

## Weekly forecast

In [86]:
df["week"] = df["date"].dt.to_period("W")

weekly = df.groupby(["week", "region", "product"], as_index=False)["utilisation"].mean()

weekly["forecast_weekly"] = weekly["utilisation"] + np.random.normal(0, 0.02, len(weekly))

## Monthly forecast

In [87]:
df["month"] = df["date"].dt.to_period("M")

monthly = df.groupby(["month", "region", "product"], as_index=False)["utilisation"].mean()

monthly["forecast_monthly"] = monthly["utilisation"] + np.random.normal(0, 0.01, len(monthly))

## Final dataset

In [88]:
df = df.merge(
    weekly[["week", "region", "product", "forecast_weekly"]],
    on=["week", "region", "product"],
    how="left"
)

df = df.merge(
    monthly[["month", "region", "product", "forecast_monthly"]],
    on=["month", "region", "product"],
    how="left"
)

df = df[['date', 'region', 'product', 'utilisation', 'forecast_weekly', 'forecast_monthly']]

df.to_csv(r"C:\Users\nikolaj\Desktop\synthetic_dataset.csv", index=False)
df.head()

,date,region,product,utilisation,forecast_weekly,forecast_monthly
0,2026-01-01,North,Mini Excavator,0.241020,0.233794,0.239343
1,2026-01-01,North,Skid Steer Loader,0.214326,0.243154,0.227714
2,2026-01-01,North,Telehandler,0.193948,0.196489,0.213634
3,2026-01-01,North,Dump Truck,0.205182,0.199795,0.175606
4,2026-01-01,North,Concrete Mixer,0.142532,0.158031,0.160006
